<a href="https://colab.research.google.com/github/GurneeshBudhiraja/ArangoDB-Hackathon/blob/main/ArangoDB_Hackathon.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Testing Application using Arango DB and Synthea Dataset using Google Colab's T4 Runtime

In [1]:
# Required packages
!pip install nx-arangodb
!pip install arango-datasets
!nvidia-smi
!nvcc --version
!pip install nx-cugraph-cu12 --extra-index-url https://pypi.nvidia.com
!pip install google-genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.8/67.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 40.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 73.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.6/114.6 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: networkx
    Found existing installation: networkx 3.4.2
    Uninstalling networkx-3.4.2:
      Successfully uninstalled networkx-3.4.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torch 2.5.1+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.5.1+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which

In [67]:
# Import required modules
import json
import pprint
from google.colab import userdata

import networkx as nx
import nx_arangodb as nxadb

from arango import ArangoClient
from arango_datasets import Datasets


# Gemini SDK Packages
from google import genai

In [68]:
# Imports the secrets from the Colab notebook
ARANGO_HOST = userdata.get("ARANGO_HOST")
ARANGO_PASSWORD = userdata.get("ARANGO_PASSWORD")
ARANGO_USERNAME = userdata.get("ARANGO_USERNAME")
GEMINI_API = userdata.get("GEMINI_API_KEY")
FLASH_MODEL = "gemini-2.0-flash"


# Creates a db connection
db = ArangoClient(hosts=ARANGO_HOST).db(username=ARANGO_USERNAME, password=ARANGO_PASSWORD, verify=True)
gemini_client = genai.Client(
    api_key=GEMINI_API
)

In [69]:
# Creates the dataset connection using the db object
datasets = Datasets(db)

DATASET_NAME = "SYNTHEA_P100"

# Conditionally Loads the Synthea P100 dataset in Arango
if not db.has_graph(DATASET_NAME):
  datasets.load(dataset_name=DATASET_NAME)
else:
  print(f"{DATASET_NAME} is already in ArangoDB.")

Output()

SYNTHEA_P100 is already in ArangoDB.


In [70]:
# Connects with the Graph in ArangoDB
graph = None
if db.has_graph(DATASET_NAME):
  graph = nxadb.Graph(name=DATASET_NAME, db=db)
else:
  print("Graph does not exist in Arango DB")

print(graph)

[00:32:38 +0000] [INFO]: Graph 'SYNTHEA_P100' exists.
INFO:nx_arangodb:Graph 'SYNTHEA_P100' exists.
[00:32:38 +0000] [INFO]: Default node type set to 'allergies'
INFO:nx_arangodb:Default node type set to 'allergies'


Graph named 'SYNTHEA_P100' with 145514 nodes and 311701 edges


In [77]:
from pydantic import BaseModel
from typing import List
# Tools for the agent

# Tool to decide whether to use AQL, NetworkX or Hybrid to resolve the user query
def decide_method(user_query:str) -> str:
  """
  This function is responsible to decide whether to use AQL or NetworkX or Combined algorithms to further answer the user query.

  Args:
    user_query: The question asked by the user.

  Returns:
    Returns a string with the prefered method to use for the user_query
  """
  print("================= USING THE decide_method TOOL =================")
  system_instruction = f"""You are an expert in choosing the best method to query a graph database.
        You must choose one of the following methods:
        - aql: For direct database queries and relationship traversals.
        - networkX: For graph algorithm calculations.
        - hybrid: For combining AQL and NetworkX.

        Return ONLY a JSON object in the following format:

        {{
        "method": "method_name"
        }}

        Where 'method_name' is one of 'aql', 'networkX', or 'hybrid'. Do NOT return anything else.
        """
  response = gemini_client.models.generate_content(
      model=FLASH_MODEL,
      config={
          "system_instruction": system_instruction
      },
      contents=[user_query]
  )
  print(response.text)
  return response.text


class AQLSchema(BaseModel):
  query:str

# Tool to generate aql queries
def aql_generator(user_query: str) -> dict[str, str]:
  """
  This is used to generate the AQL queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' and AQL query which will be a string
  """
  print("================= USING THE aql_generator TOOL =================")
  aql_generator_system_prompt = """
  You are the ai assistant who will generate AQL queries. The ArangoDB has Synthea_P100 dataset with the following vertex and edge collections:

  Vertex Collections:
    allergies
    careplans
    conditions
    devices
    encounters
    imaging_studies
    immunizations
    medications
    observations
    organizations
    patients
    payers
    procedures
    providers
    supplies

  Edge Collections:
    encounters_to_allergies
    encounters_to_careplans
    encounters_to_conditions
    encounters_to_devices
    encounters_to_imaging_studies
    encounters_to_immunizations
    encounters_to_medications
    encounters_to_observations
    encounters_to_procedures
    encounters_to_supplies
    organizations_to_encounters
    organizations_to_providers
    patients_to_allergies
    patients_to_careplans
    patients_to_conditions
    patients_to_devices
    patients_to_encounters
    patients_to_imaging_studies
    patients_to_immunizations
    patients_to_medications
    patients_to_observations
    patients_to_procedures
    patients_to_supplies
    payers_to_encounters
    payers_to_medications
    providers_to_encounters

  Your job is to generate an AQL query based on the user's query. Make sure the query is correct and achieves the purpose and gets the info that is needed. Make sure the AQL query should work on the Synthea_P100 dataset and make sure that the collection names, field names, and any other thing you mention is compatible with the dataset that is in the ArangoDB dataset.
  """
  response = gemini_client.models.generate_content(
      model="gemini-1.5-pro",
      config={
          "system_instruction": aql_generator_system_prompt,
          "response_mime_type": "application/json",
          "response_schema":AQLSchema
      },
      contents=[user_query]
  )
  return (json.loads(response.text))


def AQL_executor(aql_query:str):
  """
  This is used to executed the queries that are in the AQL format using the arangoDB client

  Args:
    aql_query: This is the AQL query that can be executed in the ArangoDB

  Returns:
    This returns the list of the data from the ArangoDB. The empty list means that nothing related to that query is available
  """
  print("================= USING THE AQL_executor TOOL =================")
  print(f"AQL Query:\n{aql_query}")
  cursor = db.aql.execute(aql_query)
  return list(cursor)


# Tool to generate networkX queries
def networkX_generator(user_query:str) -> dict[str, str]:
  """
  This is used to generate the NetworkX queries using the user query

  Args:
    user_query: The query asked by the user

  Returns:
    Returns a dict with a key 'query' which will be NetworkX query which will be a string
  """
  print("================= USING THE networkX_generator TOOL =================")
  return {"query":"This is a sample networkX query"}

class ChunkerSchema(BaseModel):
  queries: str

def chunker(user_query:str)->list[str]:
  """
  The tool is used to divide the user query into simple chunks(queries) for further tools

  Args:
  user_query: The initial user_query entered by the user

  Returns:
  The list of strings that contains queries for further tools are sent
  """
  print("================= USING THE chunker TOOL =================")

  chunker_prompt= """
  Your job is to take in the user query and break into the list of simple queries for easy retrieval. For instance, if the initial user queries ask about so many data from the db break those queries into simple queries with proper data so that at the end we can get collective data for the user queries for the AI model to use this to answer the user's initial query.
  Note: You will be dividing the chunks for the Synthea_P100 dataset stored in the ArangoDb. Your only job is to break the query into the list of simple queries.

  Keep in mind your job is not to create any queries but rather conver the user query into simple natural queries as if you are the human who is modifying the complex natural language queries into simple natural language queries for the further tools to act upon.
  """
  json_repsonse = gemini_client.models.generate_content(
      model=FLASH_MODEL,
      config={
          "system_instruction": chunker_prompt,
          "response_mime_type": "application/json",
          "response_schema":list[str]
      },
      contents=[user_query]
  )
  response = json.loads(json_repsonse.text)
  print(f"================= The new query is =================\n{response}")
  return response

In [100]:
tools = [decide_method,aql_generator,networkX_generator, AQL_executor,chunker]
system_instructions = """You are the smart agent whose main job is to use the required tools to help answer the user query when the data that is stored is Synthea_P100. These are the tools you have access to:

Please make sure to always run the chunker tool that would help break the complex user queries into simple natural language queries. The chunker tool would return list of queries that you have to run all of them using the given tools. Also, find the best method and solve and get answer for a single query at a time before going to the next one.

At the end, for all the data that you collected from different tools during the opearation, combine that result into the natural language and present to the user. Make sure the final response should give an idea at the end what sort data has been fetched by you and what data is missing.
"""

In [101]:

config = {
    'tools': tools,
    'system_instruction': system_instructions,
}


In [105]:
user_input = input("Enter your query: ").strip()
if not user_input:
  user_input = "What are the allergies and the first name of the patient with the patient id patients/1ad83dcf-7377-a7af-0063-93ee5e3e0383"
agent_response = gemini_client.models.generate_content(
    model=FLASH_MODEL,
    config=config,
    contents=[user_input]
)
print(f"Agent response is:")
try:
  print(agent_response.text)
except:
  print(agent_response)

Enter your query: This is the patient id: patients/6b031df5-bade-5f6b-6258-d9bcb897cf55 and your job is to tell what are the allergies associated with this patient and from the allergies data get me the CODE for the allergy
================= USING THE chunker TOOL =================
================= The new query is =================
['What are the allergies associated with patient ID patients/6b031df5-bade-5f6b-6258-d9bcb897cf55?', 'Get the CODE for each allergy.']
================= USING THE decide_method TOOL =================
```json
{
"method": "aql"
}
```
================= USING THE aql_generator TOOL =================
================= USING THE AQL_executor TOOL =================
AQL Query:
FOR v IN allergies FILTER v._id IN (FOR vv IN patients_to_allergies FILTER vv._from == "patients/6b031df5-bade-5f6b-6258-d9bcb897cf55" RETURN vv._to) RETURN v
================= USING THE decide_method TOOL =================
```json
{
"method": "aql"
}
```
================= USING THE aql_gene